# 🤖 AI Chatbot Executive Dashboard — Powered by Claude Sonnet

Interactive AI assistant answering business questions from live data across all 6 Gold tables.

## Setup Required
Add to `/Workspace/ecommerce_retail/config.py`:
```python
ANTHROPIC_API_KEY = "sk-ant-..."   # your Anthropic API key
```
Then **Run All** cells top-to-bottom.

In [ ]:
exec(open('/Workspace/ecommerce_retail/config.py').read())

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import requests, warnings
warnings.filterwarnings('ignore')

def read_gold(t):
    return (spark.read
        .option('fs.s3a.access.key', ACCESS_KEY)
        .option('fs.s3a.secret.key', SECRET_KEY)
        .option('fs.s3a.session.token', SESSION_TOKEN)
        .option('fs.s3a.aws.credentials.provider',
                'org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider')
        .parquet(f'{S3_GOLD_BASE}/{t}/')
        .toPandas())

print('⏳ Loading Gold tables...')
df_sales     = read_gold('sales_summary')
df_customers = read_gold('customer_analytics')
df_products  = read_gold('product_performance')
df_inventory = read_gold('inventory_health')
df_payments  = read_gold('payment_summary')
df_reviews   = read_gold('review_analytics')
print('✅ All 6 Gold tables loaded')

In [ ]:
# ── Build Business Context ────────────────────────────────────────────────
def build_context():
    # Sales
    total_net_rev = df_sales['net_revenue'].sum()
    total_orders  = df_sales['total_orders'].sum()
    avg_aov       = df_sales['avg_order_value'].mean()
    avg_fulfill   = df_sales['avg_fulfillment_days'].mean()
    rev_by_year   = df_sales.groupby('order_year')['net_revenue'].sum().to_dict()
    # Customers
    total_cust    = len(df_customers)
    avg_ltv       = df_customers['lifetime_value'].mean()
    seg_counts    = df_customers['customer_segment'].value_counts().to_dict()
    country_cts   = df_customers['country'].value_counts().to_dict()
    seg_ltv       = df_customers.groupby('customer_segment')['lifetime_value'].mean().round(0).to_dict()
    country_ltv   = df_customers.groupby('country')['lifetime_value'].mean().sort_values(ascending=False).round(0).to_dict()
    status_cts    = df_customers['customer_status'].value_counts().to_dict()
    pay_pref      = df_customers['preferred_payment_method'].value_counts().to_dict()
    seg_ctry      = df_customers.groupby(['country','customer_segment']).size().unstack(fill_value=0).to_dict()
    # Products
    total_prod    = len(df_products)
    rev_by_cat    = df_products.groupby('category')['revenue'].sum().sort_values(ascending=False).round(0).to_dict()
    profit_by_cat = df_products.groupby('category')['profit_estimate'].sum().sort_values(ascending=False).round(0).to_dict()
    units_by_cat  = df_products.groupby('category')['units_sold'].sum().sort_values(ascending=False).to_dict()
    margin_by_cat = df_products.groupby('category').apply(
        lambda x: round(x['profit_estimate'].sum()/x['revenue'].sum()*100,1) if x['revenue'].sum()>0 else 0
    ).to_dict()
    top5          = df_products.nlargest(5,'revenue')[['product_name','category','revenue','profit_estimate','avg_review_rating']].round(2).to_dict('records')
    low_rated     = df_products.nsmallest(5,'avg_review_rating')[['product_name','category','avg_review_rating','units_sold']].round(2).to_dict('records')
    # Inventory
    total_sku     = len(df_inventory)
    inv_st        = df_inventory['inventory_status'].value_counts().to_dict()
    ls_pct        = df_inventory['low_stock_flag'].sum()/total_sku*100
    by_ctry       = df_inventory.groupby('warehouse_country')['available_stock'].sum().to_dict()
    low_ctry      = df_inventory[df_inventory['low_stock_flag']==True].groupby('warehouse_country').size().to_dict()
    sor           = df_inventory[df_inventory['available_stock']<df_inventory['reorder_point']].groupby('warehouse_country').size().to_dict()
    total_inv_val = df_inventory['inventory_value'].sum()
    crit          = df_inventory.nsmallest(5,'available_stock')[['product_name','warehouse_country','available_stock','reorder_point','inventory_status']].to_dict('records')
    # Payments
    total_txns    = df_payments['transaction_count'].sum()
    suc_rate      = df_payments[df_payments['payment_status']=='Success']['transaction_count'].sum()/total_txns*100
    pay_st        = df_payments.groupby('payment_status')['transaction_count'].sum().to_dict()
    pay_vol       = df_payments.groupby('payment_method')['transaction_count'].sum().sort_values(ascending=False).to_dict()
    pay_amt       = df_payments.groupby('payment_method')['total_amount'].sum().sort_values(ascending=False).round(0).to_dict()
    fraud_m       = df_payments.groupby('payment_method')['avg_fraud_score'].mean().sort_values(ascending=False).round(3).to_dict()
    failed_m      = df_payments[df_payments['payment_status']=='Failed'].groupby('payment_method')['transaction_count'].sum().sort_values(ascending=False).to_dict()
    # Reviews
    df_ru         = df_reviews.drop_duplicates(subset=['review_id']) if 'review_id' in df_reviews.columns else df_reviews
    df_rp         = df_reviews.drop_duplicates(subset=['product_id'])
    avg_r         = df_reviews['avg_rating'].mean()
    sent          = df_ru['sentiment'].value_counts().to_dict()
    nps           = df_ru['nps_category'].value_counts().to_dict()
    trd           = df_ru['rating_trend'].value_counts().to_dict()
    cat_r         = df_rp.groupby('category')['avg_rating'].mean().sort_values(ascending=False).round(2).to_dict()
    pos_r         = df_reviews['positive_rate_pct'].mean()
    neg_r         = df_reviews['negative_rate_pct'].mean()

    parts = [
        'ECOMMERCE RETAIL LAKEHOUSE - LIVE BUSINESS DATA',
        '='*55,
        'SCOPE: 5 countries | 4 segments | 6 payment methods',
        '',
        '=== SALES ===',
        f'Total Net Revenue     : ${total_net_rev:,.0f}',
        f'Total Orders          : {total_orders:,}',
        f'Avg Order Value       : ${avg_aov:,.0f}',
        f'Avg Fulfillment Days  : {avg_fulfill:.1f}',
        f'Revenue by Year       : {rev_by_year}',
        '',
        '=== CUSTOMERS ===',
        f'Total Customers       : {total_cust:,}',
        f'Status                : {status_cts}',
        f'Segments              : {seg_counts}',
        f'Countries             : {country_cts}',
        f'Avg Lifetime Value    : ${avg_ltv:,.0f}',
        f'LTV by Segment        : {seg_ltv}',
        f'LTV by Country        : {country_ltv}',
        f'Preferred Payment     : {pay_pref}',
        f'Segment x Country     : {seg_ctry}',
        '',
        '=== PRODUCTS ===',
        f'Total Products        : {total_prod:,}',
        f'Revenue by Category   : {rev_by_cat}',
        f'Profit by Category    : {profit_by_cat}',
        f'Units Sold by Cat     : {units_by_cat}',
        f'Margin % by Cat       : {margin_by_cat}',
        f'Top 5 Products        : {top5}',
        f'Lowest-Rated Products : {low_rated}',
        '',
        '=== INVENTORY ===',
        f'Total SKUs            : {total_sku:,}',
        f'Status Mix            : {inv_st}',
        f'Low Stock %           : {ls_pct:.1f}%',
        f'Total Inventory Value : ${total_inv_val:,.0f}',
        f'Avail Stock/Country   : {by_ctry}',
        f'Low Stock/Country     : {low_ctry}',
        f'Below Reorder Point   : {sor}',
        f'5 Critical SKUs       : {crit}',
        '',
        '=== PAYMENTS ===',
        f'Total Transactions    : {total_txns:,}',
        f'Success Rate          : {suc_rate:.1f}%',
        f'Status Breakdown      : {pay_st}',
        f'Volume by Method      : {pay_vol}',
        f'Amount by Method      : {pay_amt}',
        f'Avg Fraud/Method      : {fraud_m}',
        f'Failed Txns/Method    : {failed_m}',
        '',
        '=== REVIEWS ===',
        f'Avg Rating            : {avg_r:.2f}/5.0',
        f'Sentiment             : {sent}',
        f'NPS                   : {nps}',
        f'Rating Trend          : {trd}',
        f'Avg Rating by Cat     : {cat_r}',
        f'Positive Rate         : {pos_r:.1f}%',
        f'Negative Rate         : {neg_r:.1f}%',
    ]
    return chr(10).join(parts)

BUSINESS_CONTEXT = build_context()
print(f'✅ Context built ({len(BUSINESS_CONTEXT):,} chars)')

In [ ]:
# ── Claude API Function ───────────────────────────────────────────────────
def get_system_prompt():
    return (
        'You are a senior Business Intelligence Analyst and strategic advisor for a global '
        'ecommerce retail company operating across Australia, Canada, India, UK, and USA.\n\n'
        'You have real-time aggregated data from the Gold layer of the data lakehouse.\n\n'
        'LIVE BUSINESS DATA:\n'
        + BUSINESS_CONTEXT +
        '\n\nRESPONSE FORMAT:\n'
        '- Lead with the direct answer and cite specific numbers\n'
        '- Use **bold** for key metrics\n'
        '- Include a Recommendations section with 3-5 prioritised actionable steps\n'
        '- Be executive-level: clear, concise, no jargon\n'
    )

def ask_claude(question, history):
    messages = history + [{'role':'user','content':question}]
    try:
        r = requests.post(
            'https://api.anthropic.com/v1/messages',
            headers={'x-api-key':ANTHROPIC_API_KEY,'anthropic-version':'2023-06-01',
                     'content-type':'application/json'},
            json={'model':'claude-sonnet-4-6','max_tokens':1500,
                  'system':get_system_prompt(),'messages':messages},
            timeout=45)
        r.raise_for_status()
        answer = r.json()['content'][0]['text']
    except requests.exceptions.Timeout:
        return '⚠️ Request timed out. Please try again.', history
    except Exception as e:
        return f'❌ Error: {str(e)}', history
    updated = messages + [{'role':'assistant','content':answer}]
    if len(updated) > 20: updated = updated[-20:]  # keep last 10 turns
    return answer, updated

print('✅ Claude API function ready')

In [ ]:
# ── Chat UI with ipywidgets ───────────────────────────────────────────────
conversation_history = []

display(HTML('''
<style>
.chat-hdr  { background:linear-gradient(135deg,#1a1a2e,#16213e); color:white;
             padding:14px 18px; border-radius:10px 10px 0 0; font-size:17px; font-weight:700; }
.user-msg  { background:#3498db; color:white; padding:10px 14px;
             border-radius:18px 18px 4px 18px; margin:8px 0 8px 80px; font-size:13px; }
.claude-msg{ background:#f7f9ff; border-left:4px solid #3498db; padding:12px 16px;
             border-radius:4px 18px 18px 4px; margin:8px 80px 8px 0;
             font-size:13px; line-height:1.6; white-space:pre-wrap; }
</style>'''))

header_w  = widgets.HTML('<div class="chat-hdr">🤖 AI Executive Assistant — Claude Sonnet</div>')
chat_log  = widgets.Output(layout=widgets.Layout(
    height='420px', overflow_y='auto', border='1px solid #dde3f0',
    padding='10px', background_color='#fafbff'))
q_box     = widgets.Textarea(
    placeholder='Ask a business question — e.g. Which payment method has the highest fraud risk?',
    layout=widgets.Layout(width='100%', height='70px'))
ask_btn   = widgets.Button(description='Ask Claude ▶', button_style='primary',
                            layout=widgets.Layout(width='130px', height='36px'))
clear_btn = widgets.Button(description='Clear ✕', button_style='warning',
                            layout=widgets.Layout(width='80px', height='36px'))
status_w  = widgets.HTML('')

QUICK_QS = [
    'How can I convert Indian Bronze customers to Gold customers?',
    'Which product category has the highest return risk?',
    'Which payment method has the highest fraud risk?',
    'Which warehouse is closest to stockout?',
    'What is driving the overall revenue trend? Give top 3 growth levers.',
    'Which customer segment has the best ROI for targeted campaigns?',
    'Give me an executive summary of the entire business health.',
]
quick_btns = [
    widgets.Button(
        description=(q if len(q)<=58 else q[:55]+'...'),
        tooltip=q,
        layout=widgets.Layout(width='100%', height='30px'),
        style={'button_color':'#eaf0fb', 'font_size':'11px'}
    ) for q in QUICK_QS
]

def show_msg(role, text):
    css = 'user-msg' if role=='user' else 'claude-msg'
    lbl = '👤 You' if role=='user' else '🤖 Claude'
    with chat_log:
        display(HTML(f'<div class="{css}"><b>{lbl}:</b><br>{text}</div>'))

def on_ask(b):
    global conversation_history
    q = q_box.value.strip()
    if not q: return
    q_box.value = ''
    show_msg('user', q)
    status_w.value = "<i style='color:#3498db;font-size:12px'>⏳ Claude is thinking…</i>"
    ask_btn.disabled = True
    try:
        answer, conversation_history = ask_claude(q, conversation_history)
        show_msg('claude', answer)
    except Exception as e:
        show_msg('claude', f'❌ Error: {e}')
    finally:
        status_w.value = ''
        ask_btn.disabled = False

def on_clear(b):
    global conversation_history
    conversation_history = []
    with chat_log: clear_output()
    status_w.value = "<i style='color:#888;font-size:12px'>🗑️ Chat cleared</i>"

def make_quick(q):
    def h(b): q_box.value = q
    return h

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)
for btn, q in zip(quick_btns, QUICK_QS):
    btn.on_click(make_quick(q))

qs_box = widgets.VBox(
    [widgets.HTML("<b style='font-size:12px;color:#555'>⚡ Quick Questions (click to load):</b>")] + quick_btns,
    layout=widgets.Layout(padding='6px 0 10px')
)
input_row = widgets.HBox(
    [q_box, widgets.VBox([ask_btn, clear_btn], layout=widgets.Layout(gap='6px'))],
    layout=widgets.Layout(gap='8px', align_items='flex-start', padding='6px 0')
)
display(widgets.VBox([
    header_w, qs_box,
    widgets.HTML("<hr style='margin:2px 0;border-color:#dde3f0'>"),
    input_row, status_w,
    widgets.HTML("<div style='font-size:11px;color:#aaa'>💬 Conversation:</div>"),
    chat_log
]))

---
### 💡 Example Questions

| Area | Question |
|---|---|
| Customer Strategy | *How can I convert Indian Bronze customers to Gold customers?* |
| Revenue Growth | *What is driving the revenue trend? Give top 3 growth levers.* |
| Fraud Risk | *Which payment method has the highest fraud risk?* |
| Inventory | *Which warehouse is closest to stockout?* |
| Product Risk | *Which product category has the highest return risk?* |
| Full Summary | *Give me an executive summary of the entire business health.* |

> The chatbot keeps conversation history — ask follow-up questions naturally.
> Use **Clear** to start a fresh session.